In [1]:
!pip install -q open_clip_torch ftfy regex tqdm pillow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00


In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("clip")

In [3]:
import os
import torch
from urllib.request import urlopen
from PIL import Image
import open_clip
from huggingface_hub import login

# ============================================================
# LOAD HF TOKEN SAFELY FROM KAGGLE SECRETS
# ============================================================

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("clip")
    print("✅ HF_TOKEN loaded from Kaggle Secrets.")
except Exception as e:
    HF_TOKEN = None
    print("⚠️ HF_TOKEN not found in Kaggle Secrets.")
    print("Reason:", e)

if HF_TOKEN:
    login(token=HF_TOKEN)
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
else:
    print("⚠️ Running without HF token. Download may be slower or fail if gated.")

# ============================================================
# LOAD BIOMEDCLIP
# ============================================================

MODEL_ID = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"

Device= torch.device("cuda")
print("Device:", Device)

model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    MODEL_ID
)

tokenizer = open_clip.get_tokenizer(MODEL_ID)

model = model.to(Device)
model.eval()


✅ HF_TOKEN loaded from Kaggle Secrets.
Device: cuda


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

CustomTextCLIP(
  (visual): TimmModel(
    (trunk): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): Identity()
          (drop_path1): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
          

In [5]:
import sys, subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/openai/CLIP.git",
    "ftfy", "regex", "tqdm"
])
import clip
print("✅ clip imported:", clip.__file__)

✅ clip imported: /usr/local/lib/python3.12/dist-packages/clip/__init__.py


In [6]:
!pip install faiss-gpu-cu12  

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 38.8 MB/s eta 0:00:00:00:0100:01


In [7]:
!pip install faiss-gpu-cu12
import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/openai/CLIP.git",
    "ftfy", "regex", "tqdm"
])

import clip

# Updated code

In [ ]:
# ================================================================================
# STAGE 2 + STAGE 3  —  v11   (v10 + CAUSAL CONTAINMENT FIX)
# ================================================================================
# ONE Kaggle cell. Executes top-to-bottom. Section banners are navigation only.
#
# WHY v11 (the only change vs v10 is the CAUSAL gate geometry)
# --------------------------------------------------------------------------------
# Diagnosis on the v10 smoke run: causal sim 0.27–0.35 (>> thresholds ≤0.18) and
# persistence_used=47 with ZERO persistence_fallback — so BOTH halves of the causal
# AND were healthy, yet 65% of causal patches still fell to fallback. Cause: the
# containment test was `persistent_mask[box_CENTER_pixel] > 0.5`. The persistence
# blob is one small component (TOP_K=1); boxes are 128/256px. A big box can cover
# the blob while its CENTER pixel lands just outside it -> not causal -> fallback.
#
# [C-FIX-1] peak-or-coverage containment: a box is causal if it contains the blob's
#           PEAK pixel OR captures >= CAUSAL_MASK_COVER_FRAC of the blob — instead of
#           requiring the box center to sit on the blob. Scale-invariant; keeps the
#           topology gate the framing/ablations depend on.
# [C-FIX-2] dilate the persistence blob by CAUSAL_MASK_DILATE_PX before testing, so
#           near-miss boxes are forgiven.
# [C-FIX-3] PERSISTENCE_TOP_K 1 -> 2, so multi-focal / larger-extent findings give a
#           bigger mask (bilateral effusion/edema were mis-served by a single blob).
#
# All v10 behaviour otherwise UNCHANGED:
#   [M-1] causal = semantic ≥ per-class thr AND (new) blob containment
#   [M-2] spur_in = non-causal in-anatomy, lowest-sim, safety-net
#   [M-3] spur_out = gate-free inverse-heatmap + intensity scan
#   [M-4] causal query = report-snippet ⊕ class-prompt blend
#   + atomic resumable checkpoints · wall-clock guard · multi-class filter · viz cap.
#
# NOTE: causal composition moves with these knobs -> bounces Cell-2 ablations.
#       Settle CAUSAL_MASK_COVER_FRAC / DILATE / TOP_K before any REPORTED run and
#       flag to Dr. Nimi.
# ================================================================================

import os, re, gc, json, time, pickle, math, warnings, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont
from tqdm import tqdm
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import Counter, defaultdict
from scipy.ndimage import label as ndimage_label, binary_dilation   # [C-FIX-2]

warnings.filterwarnings("ignore")

_T_START = time.time()

# ================================================================================
# SECTION 0 — CONFIG
# ================================================================================

assert torch.cuda.is_available(), "❌ CUDA required."
Device = torch.device("cuda")

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.set_float32_matmul_precision("high")

print(f"✅ GPU : {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

CSV_DIR       = "/kaggle/input/combinedreportimgpath"
TRAIN_CSV     = f"{CSV_DIR}/train_pairs_labeled.txt"
VAL_CSV       = f"{CSV_DIR}/val_pairs_labeled.txt"
TEST_CSV      = f"{CSV_DIR}/test_pairs_labeled.txt"
CHEXZERO_CKPT = ("/kaggle/input/models/anikazarin/chexzero/pytorch/default/1/"
                 "best_64_5e-05_original_22000_0.864.pt")
OUT_DIR = "/kaggle/working"
VIS_DIR = f"{OUT_DIR}/vis"
os.makedirs(OUT_DIR, exist_ok=True)
for sp in ("train", "val"):
    os.makedirs(f"{VIS_DIR}/{sp}", exist_ok=True)

STAGE2_OUT       = f"{OUT_DIR}/stage2_outputs.pkl"
STAGE3_RESULTS   = f"{OUT_DIR}/stage3_outputs.pkl"
STAGE3_CAUSAL_PT = f"{OUT_DIR}/stage3_causal.pt"
STAGE3_SPIN_PT   = f"{OUT_DIR}/stage3_spur_in.pt"
STAGE3_SPOUT_PT  = f"{OUT_DIR}/stage3_spur_out.pt"
STAGE3_META      = f"{OUT_DIR}/stage3_meta.pkl"

TARGET_CLASSES = ["Atelectasis", "Cardiomegaly", "Consolidation",
                  "Edema", "Pleural Effusion"]

SMOKE_TEST             = False
SMOKE_TRAIN_N          = 40
SMOKE_VAL_N            = 20
MULTICLASS_ONLY        = True
FAST_MODE              = False
MAX_TRAIN_IMAGES       = None
MAX_VAL_IMAGES         = 2000
USE_LABEL_BACKUP_TRAIN = False
USE_LABEL_BACKUP_VAL   = False

WALLCLOCK_BUDGET_HOURS = 9.0
CKPT_EVERY_IMAGES      = 500
CKPT_EVERY_MIN         = 12
VIS_MAX_PER_SPLIT      = 150

IMAGE_SIZE             = 512
PATCH_ENCODE_BATCH_SZ  = 128
TEXT_ENCODE_BATCH_SZ   = 64
ENABLE_GRADCAM         = True
ENABLE_ZOOM_REFINEMENT = True
ENABLE_VISUALIZATION   = True

MAX_ITER                 = 2 if FAST_MODE else 3
MAX_CANDIDATE_BOXES      = 100 if FAST_MODE else 160
MAX_ZOOM_CANDIDATE_BOXES = 80  if FAST_MODE else 120

SCORE_W_SEM  = 0.45
SCORE_W_PROB = 0.35
SCORE_W_GC   = 0.20

SEMANTIC_THRESHOLD = 0.16
SEMANTIC_THRESHOLD_PER_CLASS = {
    "Atelectasis": 0.16, "Cardiomegaly": 0.18, "Consolidation": 0.14,
    "Edema": 0.15, "Pleural Effusion": 0.18,
}
GRADCAM_WEAK_THR   = 0.08
SPATIAL_THRESH     = 0.18
CONF_THRESHOLD     = 0.30 if FAST_MODE else 0.40
FALLBACK_TOP_K     = 2
MIN_CAUSAL_PATCHES = 2
TOP_K_PER_FIND     = 3 if FAST_MODE else 5
STRIDE_BASE        = 32
PATCH_SCALES       = [128, 256] if FAST_MODE else [64, 128, 256]
ZOOM_SCALES        = [64] if FAST_MODE else [48, 64, 96]
GRADCAM_SNIPPET_W  = 0.65
GRADCAM_CLASS_W    = 0.35

# ── [C-FIX-1/2] Causal containment (fixes brittle center-pixel test → 65% fallback)
# "center"       : legacy v10 (box center pixel must be inside blob)  [DON'T use]
# "peak"         : box must contain the blob PEAK pixel (or center-in)
# "cover"        : box must capture >= COVER_FRAC of the blob (or center-in)
# "peak_or_cover": either peak-containment OR coverage (recommended default)
CAUSAL_CONTAIN_MODE    = "peak_or_cover"
CAUSAL_MASK_COVER_FRAC = 0.35   # fraction of blob pixels the box must contain
CAUSAL_MASK_DILATE_PX  = 8      # dilate blob before testing (0 = off)

SPUR_IN_MAX_KEEP   = TOP_K_PER_FIND
SPUR_IN_GC_FLOOR   = GRADCAM_WEAK_THR

CAUSAL_QUERY_CLASS_BLEND = 0.35

_SCALE_MAP = {"Atelectasis": [64, 128], "Cardiomegaly": [128, 256],
              "Consolidation": [128, 256], "Edema": [64, 128],
              "Pleural Effusion": [128, 256]}

ZS_PROMPTS = {
    "Atelectasis": ["atelectasis on chest x-ray", "lung collapse on chest radiograph",
                    "plate-like atelectasis in the lung", "subsegmental atelectasis chest x-ray"],
    "Cardiomegaly": ["cardiomegaly on chest x-ray", "enlarged cardiac silhouette radiograph",
                     "cardiac enlargement chest radiograph", "increased cardiothoracic ratio on x-ray"],
    "Consolidation": ["consolidation on chest x-ray", "airspace opacity in the lung",
                      "lobar consolidation on radiograph", "air bronchogram in consolidated lung"],
    "Edema": ["pulmonary edema on chest x-ray", "bilateral interstitial edema radiograph",
              "vascular congestion in both lungs", "perihilar edema on chest radiograph"],
    "Pleural Effusion": ["pleural effusion on chest x-ray", "blunting of costophrenic angle",
                         "pleural fluid on chest radiograph", "layering pleural effusion x-ray"],
}

PERSISTENCE_N_LEVELS = 32
PERSISTENCE_TOP_K    = 2        # [C-FIX-3] was 1 — multi-focal / larger extent
PERSISTENCE_MIN_AREA = 16
PERSISTENCE_ENABLED  = True

OUTSIDE_ANAT_SCALES     = [64, 96, 128]
OUTSIDE_ANAT_INV_THRESH = 0.55
OUTSIDE_ANAT_MAX_CAND   = 120
OUTSIDE_ANAT_SIM_CAP    = 0.28
OUTSIDE_ANAT_MAX_KEEP   = 6

ARTIFACT_BORDER_FRAC = 0.15
ARTIFACT_MEAN_LOW    = 0.12
ARTIFACT_STD_HIGH    = 0.15
ARTIFACT_SCALES      = [64, 96]
ARTIFACT_SIM_CAP     = 0.28
ARTIFACT_MAX_KEEP    = 4

_COUNTERS = defaultdict(int)

_DEADLINE = _T_START + WALLCLOCK_BUDGET_HOURS * 3600.0
def _budget_left_h():
    return max(0.0, (_DEADLINE - time.time()) / 3600.0)

def _atomic_pickle(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(obj, f, protocol=4)
    os.replace(tmp, path)

# ================================================================================
# SECTION 1 — DATACLASSES
# ================================================================================

@dataclass
class AnatomicalPrior:
    region_name: str; center_x: float; center_y: float
    sigma_x: float; sigma_y: float

@dataclass
class PathologyQueryItem:
    pathology: str; text_snippet: str
    query_vector: Optional[torch.Tensor] = None
    anatomical_prior: Optional[AnatomicalPrior] = None
    confidence: float = 1.0; negated: bool = False; source: str = "rule"

@dataclass
class QueryPlan:
    image_name: str; image_path: str; report_path: str
    view_position: str; findings_text: str; impression_text: str
    query_items: List[PathologyQueryItem] = field(default_factory=list)
    suppressed_regions: List[str] = field(default_factory=list)

@dataclass
class PatchDocument:
    image_name: str; pathology: str; scale: int
    box: Tuple[int, int, int, int]; visual_embedding: torch.Tensor
    semantic_score: float; zeroshot_prob: float
    gradcam_score: float; combined_score: float; causal: bool
    anatomical_region: str; confidence: float; text_snippet: str
    zoom_level: int = 1; spurious_source: str = "in_anatomy"
    selection_source: str = "threshold"

@dataclass
class ImagePatchResult:
    image_name: str; image_path: str; split: str
    causal_patches: List[PatchDocument] = field(default_factory=list)
    spurious_patches: List[PatchDocument] = field(default_factory=list)
    refined: bool = False; n_iterations: int = 1; used_fallback: bool = False

# ================================================================================
# SECTION 2 — ANATOMY COORDS + GPU HEATMAPS
# ================================================================================

ANATOMY_COORDS_PA = {
    "right lung": (0.72, 0.42, 0.14, 0.22), "left lung": (0.28, 0.42, 0.14, 0.22),
    "bilateral lungs": (0.50, 0.42, 0.38, 0.22), "lungs": (0.50, 0.42, 0.38, 0.22),
    "lung": (0.50, 0.42, 0.38, 0.22),
    "right upper lobe": (0.72, 0.18, 0.10, 0.10), "right middle lobe": (0.72, 0.38, 0.10, 0.09),
    "right lower lobe": (0.72, 0.60, 0.10, 0.11), "left upper lobe": (0.28, 0.18, 0.10, 0.10),
    "left lower lobe": (0.28, 0.60, 0.10, 0.11),
    "lower lobes": (0.50, 0.62, 0.32, 0.10), "upper lobes": (0.50, 0.16, 0.32, 0.09),
    "lower lobe": (0.50, 0.62, 0.32, 0.10),
    "right costophrenic angle": (0.78, 0.81, 0.07, 0.06),
    "left costophrenic angle": (0.22, 0.81, 0.07, 0.06),
    "costophrenic angles": (0.50, 0.81, 0.36, 0.06),
    "costophrenic angle": (0.50, 0.81, 0.36, 0.06),
    "costophrenic": (0.50, 0.81, 0.36, 0.06),
    "cardiac silhouette": (0.47, 0.48, 0.12, 0.14),
    "heart": (0.47, 0.48, 0.12, 0.14),
    "cardiomegaly": (0.47, 0.48, 0.12, 0.14),
    "mediastinum": (0.50, 0.34, 0.08, 0.18), "mediastinal": (0.50, 0.34, 0.08, 0.18),
    "hila": (0.50, 0.42, 0.08, 0.08), "right hilum": (0.61, 0.42, 0.05, 0.06),
    "left hilum": (0.40, 0.42, 0.05, 0.06), "hilar": (0.50, 0.42, 0.08, 0.08),
    "apices": (0.50, 0.07, 0.26, 0.05), "right apex": (0.72, 0.06, 0.08, 0.04),
    "left apex": (0.28, 0.06, 0.08, 0.04), "apex": (0.50, 0.07, 0.26, 0.05),
    "right pleura": (0.83, 0.44, 0.05, 0.26), "left pleura": (0.17, 0.44, 0.05, 0.26),
    "pleural space": (0.50, 0.44, 0.42, 0.26), "pleural": (0.50, 0.44, 0.42, 0.26),
    "right hemidiaphragm": (0.70, 0.77, 0.12, 0.04),
    "left hemidiaphragm": (0.30, 0.77, 0.12, 0.04),
    "diaphragm": (0.50, 0.77, 0.32, 0.04), "trachea": (0.50, 0.20, 0.04, 0.08),
    "carina": (0.50, 0.30, 0.04, 0.03), "ribs": (0.50, 0.45, 0.40, 0.26),
    "clavicles": (0.50, 0.09, 0.30, 0.03), "spine": (0.50, 0.45, 0.04, 0.30),
}
ANATOMY_COORDS_AP = {
    **ANATOMY_COORDS_PA,
    "cardiac silhouette": (0.47, 0.52, 0.16, 0.17),
    "heart": (0.47, 0.52, 0.16, 0.17),
    "cardiomegaly": (0.47, 0.52, 0.16, 0.17),
    "right upper lobe": (0.71, 0.20, 0.12, 0.12), "left upper lobe": (0.29, 0.20, 0.12, 0.12),
    "right lower lobe": (0.71, 0.62, 0.11, 0.12), "left lower lobe": (0.29, 0.62, 0.11, 0.12),
}
_ANATOMY_SORTED = sorted(ANATOMY_COORDS_PA.keys(), key=len, reverse=True)

def get_anatomy_coords(region, view="PA"):
    lut = ANATOMY_COORDS_AP if str(view).upper() == "AP" else ANATOMY_COORDS_PA
    key = str(region).lower().strip()
    if key in lut:
        return lut[key]
    for k in _ANATOMY_SORTED:
        if k in key:
            return lut[k]
    return (0.50, 0.42, 0.36, 0.24)

def make_gaussian_heatmap_gpu(cx, cy, sx, sy, H=IMAGE_SIZE, W=IMAGE_SIZE, device=None):
    if device is None:
        device = Device
    xs = torch.linspace(0, 1, W, device=device)
    ys = torch.linspace(0, 1, H, device=device)
    yg, xg = torch.meshgrid(ys, xs, indexing="ij")
    return torch.exp(-((xg - cx) ** 2 / (2 * sx ** 2) + (yg - cy) ** 2 / (2 * sy ** 2))).float()

def get_composite_heatmap_gpu(plan, H=IMAGE_SIZE, W=IMAGE_SIZE, device=None):
    if device is None:
        device = Device
    composite = torch.zeros(H, W, device=device)
    suppressed = torch.zeros(H, W, device=device)
    for item in plan.query_items:
        if item.anatomical_prior is None:
            continue
        p = item.anatomical_prior
        composite += item.confidence * make_gaussian_heatmap_gpu(
            p.center_x, p.center_y, p.sigma_x, p.sigma_y, H, W, device)
    for region in plan.suppressed_regions:
        rx, ry, rsx, rsy = get_anatomy_coords(region, plan.view_position)
        suppressed += make_gaussian_heatmap_gpu(rx, ry, rsx, rsy, H, W, device)
    suppressed = suppressed.clamp(0, 1)
    maxc = composite.max()
    if maxc > 0:
        composite = composite / maxc
    return (composite * (1.0 - suppressed)).float()

# ================================================================================
# SECTION 3 — REPORT PARSING (rule-based only)
# ================================================================================

_FINDINGS_PAT   = re.compile(r"^\s*(?:4\s*\)\s*)?FINDINGS\s*:?",   re.I | re.M)
_IMPRESSION_PAT = re.compile(r"^\s*(?:5\s*\)\s*)?IMPRESSION\s*:?", re.I | re.M)
_NEG_CUES = re.compile(
    r"\b(no\b|not\b|none\b|without|absent|clear\b|unremarkable|normal\b|"
    r"no evidence of|no acute|free of|negative for|unlikely|no significant|"
    r"no definite|is not seen|are not seen)\b", re.I)
_UNC_CUES = re.compile(
    r"\b(possible|possibly|probable|probably|may|might|suspected|"
    r"cannot exclude|cannot rule out|question of|questionable|"
    r"differential|versus|vs\.?)\b", re.I)

def parse_report_sections(raw):
    lines = str(raw).split("\n")
    sections = {"findings": "", "impression": ""}
    current, buf = None, []
    for line in lines:
        if _FINDINGS_PAT.search(line):
            if current and buf:
                sections[current] = "\n".join(buf).strip()
            current, buf = "findings", []
        elif _IMPRESSION_PAT.search(line):
            if current and buf:
                sections[current] = "\n".join(buf).strip()
            current, buf = "impression", []
        elif current:
            buf.append(line)
    if current and buf:
        sections[current] = "\n".join(buf).strip()
    return sections

def sentence_status(sentence, keyword):
    sl = str(sentence).lower(); kw = str(keyword).lower(); pos = sl.find(kw)
    if pos < 0:
        return "absent"
    prefix = sl[:pos]; suffix = sl[pos:]
    if _UNC_CUES.search(prefix):
        return "uncertain"
    if _NEG_CUES.search(prefix) or _NEG_CUES.search(suffix[:50]):
        return "negated"
    return "positive"

def extract_keyword_sentences(text, keywords):
    results = []
    for sent in re.split(r"[.\n;]", str(text)):
        sent = re.sub(r"\s+", " ", sent).strip()
        if len(sent) < 8:
            continue
        for kw in keywords:
            if kw.lower() in sent.lower():
                results.append((sent, sentence_status(sent, kw)))
                break
    return results

def extract_anatomy_mentions(text):
    tl = str(text).lower()
    return [k for k in _ANATOMY_SORTED if k in tl]

SEED_KEYWORDS = {
    "Atelectasis": ["atelectasis", "atelectatic", "volume loss", "plate-like",
                    "discoid", "linear opacity", "subsegmental"],
    "Cardiomegaly": ["cardiomegaly", "enlarged cardiac", "cardiac enlargement",
                     "cardiothoracic ratio", "cardiac silhouette is enlarged"],
    "Consolidation": ["consolidation", "airspace opacity", "lobar opacity",
                      "air bronchogram", "airspace disease"],
    "Edema": ["edema", "oedema", "pulmonary congestion", "vascular congestion",
              "kerley", "perihilar", "interstitial markings", "vascular redistribution"],
    "Pleural Effusion": ["effusion", "pleural fluid", "blunting", "meniscus",
                         "costophrenic", "pleural collection", "layering"],
}
DEFAULT_ANATOMY = {"Atelectasis": "lower lobes", "Cardiomegaly": "cardiac silhouette",
                   "Consolidation": "right lower lobe", "Edema": "bilateral lungs",
                   "Pleural Effusion": "right costophrenic angle"}

def _read_report(path):
    try:
        if isinstance(path, str) and os.path.exists(path):
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()
    except Exception:
        _COUNTERS["reports_read_error"] += 1
    return ""

def _clinical_text_from_raw(raw):
    secs = parse_report_sections(raw)
    findings = secs.get("findings", ""); impression = secs.get("impression", "")
    clinical = (findings + "\n" + impression).strip()
    if len(clinical) < 20:
        clinical = str(raw)
    return findings, impression, clinical

def _rule_based_items(clinical, vocab, view, positive_labels, use_label_backup):
    disease_hits = {}
    for disease, entry in vocab.items():
        keywords = entry.get("keywords", SEED_KEYWORDS.get(disease, []))
        disease_hits[disease] = extract_keyword_sentences(clinical, keywords)

    suppressed = []
    for disease, hits in disease_hits.items():
        for sent, status in hits:
            if status == "negated":
                for region in extract_anatomy_mentions(sent):
                    if region not in suppressed:
                        suppressed.append(region)

    report_detected = {d for d, hits in disease_hits.items()
                       if any(s == "positive" for _, s in hits)}
    candidates = set(report_detected)
    if use_label_backup:
        candidates = candidates | set(positive_labels)

    items = []
    for disease in sorted(candidates):
        if disease not in vocab:
            continue
        entry = vocab[disease]; hits = disease_hits.get(disease, [])
        pos_hits = [s for s, st in hits if st == "positive"]
        unc_hits = [s for s, st in hits if st == "uncertain"]
        if pos_hits:
            snippet = max(pos_hits, key=len)
            confidence = 1.00 if (use_label_backup and disease in positive_labels) else 0.80
            negated = False
        elif unc_hits:
            snippet = max(unc_hits, key=len); confidence = 0.55; negated = False
        elif disease in positive_labels and use_label_backup:
            snippet = entry.get("query_phrases", [f"{disease.lower()} on chest x-ray"])[0]
            confidence = 0.70; negated = False
        else:
            continue
        anatomy_hits = extract_anatomy_mentions(snippet)
        primary = anatomy_hits[0] if anatomy_hits else entry.get(
            "default_anatomy", DEFAULT_ANATOMY.get(disease, "lungs"))
        cx, cy, sx, sy = get_anatomy_coords(primary, view)
        items.append(PathologyQueryItem(
            pathology=disease, text_snippet=snippet, query_vector=None,
            anatomical_prior=AnatomicalPrior(primary, cx, cy, sx, sy),
            confidence=confidence, negated=negated, source="rule"))
    return items, suppressed

# ================================================================================
# SECTION 4 — VOCAB MINING
# ================================================================================

def mine_vocab_from_reports(train_df, top_n=8, min_count=3, max_phrase_len=120):
    if "report_path" not in train_df.columns:
        print("⚠️ report_path missing — seed vocab only.")
        return {d: {"query_phrases": [f"{d.lower()} on chest x-ray"],
                    "keywords": SEED_KEYWORDS[d], "default_anatomy": DEFAULT_ANATOMY[d]}
                for d in TARGET_CLASSES}
    report_paths = train_df["report_path"].dropna().astype(str).unique().tolist()
    print(f"\n{'='*70}\nVOCAB MINER: {len(report_paths):,} reports\n{'='*70}")
    disease_sentences = {d: [] for d in TARGET_CLASSES}; skipped = 0
    for path in tqdm(report_paths, desc="Mining vocab"):
        raw = _read_report(path)
        if not raw.strip():
            skipped += 1; continue
        _, _, clinical = _clinical_text_from_raw(raw)
        for disease in TARGET_CLASSES:
            for sent_raw in re.split(r"[.\n;]", clinical):
                sent = re.sub(r"\s+", " ", sent_raw.strip())
                kw = next((k for k in SEED_KEYWORDS[disease] if k.lower() in sent.lower()), None)
                if kw and sentence_status(sent, kw) == "positive" and 15 <= len(sent) <= max_phrase_len:
                    disease_sentences[disease].append(sent.lower())
    print(f"Skipped: {skipped:,}")
    mined = {}
    for d in TARGET_CLASSES:
        sents = disease_sentences[d]
        print(f"  {d:<20}: {len(sents):,}")
        if not sents:
            mined[d] = {"query_phrases": [f"{d.lower()} on chest x-ray"],
                        "keywords": SEED_KEYWORDS[d], "default_anatomy": DEFAULT_ANATOMY[d]}
            continue
        ctr = Counter(sents)
        top = [s for s, _ in sorted([(s, c) for s, c in ctr.items() if c >= min_count and len(s) >= 20],
               key=lambda x: (-x[1], -len(x[0])))[:top_n]]
        if len(top) < 2:
            top = [s for s, _ in ctr.most_common(top_n)]
        mined[d] = {"query_phrases": top[:top_n], "keywords": SEED_KEYWORDS[d],
                    "default_anatomy": DEFAULT_ANATOMY[d]}
    return mined

# ================================================================================
# SECTION 5 — QUERY PLAN BUILDER (+ multi-class row filter)
# ================================================================================

def _detect_csv_sep(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        line = f.readline()
    return "\t" if "\t" in line and line.count("\t") >= line.count(",") else ","

def read_pairs_csv(path):
    return pd.read_csv(path, sep=_detect_csv_sep(path))

_LABEL_ALIASES = {
    "Atelectasis":      ["Atelectasis"],
    "Cardiomegaly":     ["Cardiomegaly"],
    "Consolidation":    ["Consolidation"],
    "Edema":            ["Edema"],
    "Pleural Effusion": ["Pleural Effusion", "Pleural_Effusion", "Effusion"],
}

def _find_label_cols(df):
    cols  = {c: c for c in df.columns}
    lower = {c.lower(): c for c in df.columns}
    found = {}
    for cls, aliases in _LABEL_ALIASES.items():
        hit = None
        for a in aliases:
            for cand in (a, a.replace(" ", "_"), a.lower(), a.replace(" ", "_").lower()):
                if cand in cols:
                    hit = cols[cand]; break
                if cand in lower:
                    hit = lower[cand]; break
            if hit:
                break
        if hit:
            found[cls] = hit
    return found

def filter_multiclass_rows(df, name):
    n0 = len(df)
    lcols = _find_label_cols(df)
    if lcols:
        mask = pd.Series(False, index=df.index)
        for _cls, col in lcols.items():
            v = pd.to_numeric(df[col], errors="coerce")
            mask = mask | (v == 1.0)
        out = df[mask].reset_index(drop=True)
        print(f"🎯 {name}: multi-class filter kept {len(out):,}/{n0:,} "
              f"(label cols: {list(lcols.values())})")
        return out
    aliases = set()
    for al in _LABEL_ALIASES.values():
        aliases |= set(al)
    for lc in ["Finding Labels", "finding_labels", "labels", "Finding_Labels", "Labels"]:
        if lc in df.columns:
            def _has(s):
                return len(set(str(s).split("|")) & aliases) > 0
            out = df[df[lc].apply(_has)].reset_index(drop=True)
            print(f"🎯 {name}: multi-class filter via pipe column '{lc}' "
                  f"kept {len(out):,}/{n0:,}")
            return out
    print(f"⚠️ {name}: no label columns found — skipping filter.")
    return df

def _get_col(row, cands, default=""):
    for c in cands:
        if c in row.index:
            return row[c]
    return default

def _get_label_val(row, label):
    for c in [label, label.replace(" ", "_"), label.lower(), label.replace(" ", "_").lower()]:
        if c in row.index:
            return row[c]
    return None

def build_query_plan_for_row(row, vocab, view, use_label_backup):
    img_name = str(_get_col(row, ["image_name", "Image Index", "image"], ""))
    img_path = str(_get_col(row, ["image_path", "path"], ""))
    rpt_path = str(_get_col(row, ["report_path", "report", "text_path"], ""))
    raw = _read_report(rpt_path)
    findings, impression, clinical = _clinical_text_from_raw(raw)
    plan = QueryPlan(image_name=img_name, image_path=img_path, report_path=rpt_path,
                     view_position=view, findings_text=findings, impression_text=impression)
    pos_labels = set()
    for d in TARGET_CLASSES:
        val = _get_label_val(row, d)
        try:
            if val is not None and not pd.isna(val) and float(val) == 1.0:
                pos_labels.add(d)
        except Exception:
            pass
    items, suppressed = _rule_based_items(clinical, vocab, view, pos_labels, use_label_backup)
    for qi in items:
        if qi.negated and qi.anatomical_prior:
            rn = qi.anatomical_prior.region_name
            if rn not in suppressed:
                suppressed.append(rn)
    plan.query_items = items; plan.suppressed_regions = suppressed
    return plan

def run_stage2(df, vocab, max_imgs, desc, use_label_backup, view_default="PA"):
    subset = df.iloc[:max_imgs] if max_imgs is not None else df
    plans = []
    for _, row in tqdm(subset.iterrows(), total=len(subset), desc=desc):
        view = str(_get_col(row, ["View_Position", "View Position", "view_position", "view"],
                            view_default)).upper()
        if view not in ("PA", "AP"):
            view = "PA"
        plans.append(build_query_plan_for_row(row, vocab, view, use_label_backup))
    return plans

def split_stats(plans, name):
    ni = sum(len(p.query_items) for p in plans)
    ne = sum(1 for p in plans if p.query_items)
    print(f"{name:<6}: {len(plans):,} | {ne:,} non-empty | {ni:,} items | "
          f"avg={ni/max(len(plans),1):.2f}")

# ================================================================================
# SECTION 6 — LOAD CHEXZERO
# ================================================================================

try:
    import clip
except Exception as e:
    raise ImportError("❌ Cannot import CLIP.") from e

def load_chexzero(ckpt_path, device):
    print("\n" + "="*70 + "\nLOADING CHEXZERO\n" + "="*70)
    assert os.path.exists(ckpt_path), f"❌ Not found: {ckpt_path}"
    model, preprocess = clip.load("ViT-B/32", device=device, jit=False)
    state = torch.load(ckpt_path, map_location=device)
    if isinstance(state, dict):
        for k in ("state_dict", "model", "model_state_dict", "net"):
            if k in state:
                state = state[k]; break
    clean = {k.replace("module.", "").replace("model.", ""): v for k, v in state.items()}
    miss, unex = model.load_state_dict(clean, strict=False)
    print(f"Missing: {len(miss)} | Unexpected: {len(unex)}")
    print("✅ CheXzero loaded." if len(miss) <= 50 else "⚠️ Many missing keys.")
    model.eval().to(device)
    for p in model.parameters():
        p.requires_grad_(False)
    for p in model.visual.transformer.resblocks[-1].parameters():
        p.requires_grad_(True)
    return model, preprocess

@torch.no_grad()
def build_class_text_matrix(model, device):
    rows = []
    for cls in TARGET_CLASSES:
        toks = clip.tokenize(ZS_PROMPTS[cls], truncate=True).to(device)
        e = model.encode_text(toks).float(); e = e / e.norm(dim=-1, keepdim=True)
        m = e.mean(0); m = m / m.norm(); rows.append(m)
    ctm = torch.stack(rows).float()
    ls = model.logit_scale.exp().detach().float()
    print(f"✅ CTM: {tuple(ctm.shape)} | logit_scale={ls.item():.2f}")
    return ctm, ls

@torch.no_grad()
def fill_query_vectors(plans, model, device, ctm=None, bs=TEXT_ENCODE_BATCH_SZ):
    """[M-4] Query vector = blend of REPORT-snippet embedding and class prompt."""
    refs, texts = [], []
    for p in plans:
        for qi in p.query_items:
            refs.append(qi); texts.append(str(qi.text_snippet).strip() or qi.pathology)
    if not texts:
        return plans
    model.eval(); all_e = []
    for i in tqdm(range(0, len(texts), bs), desc="Text encode"):
        toks = clip.tokenize(texts[i:i+bs], truncate=True).to(device)
        e = model.encode_text(toks).float(); e = e / e.norm(dim=-1, keepdim=True)
        all_e.append(e.detach().cpu())
    all_e = torch.cat(all_e)
    ctm_cpu = ctm.detach().cpu().float() if ctm is not None else None
    a = float(CAUSAL_QUERY_CLASS_BLEND)
    n_blend = 0
    for qi, e in zip(refs, all_e):
        e = e.float()
        if ctm_cpu is not None and a > 0.0 and qi.pathology in TARGET_CLASSES:
            cprompt = ctm_cpu[TARGET_CLASSES.index(qi.pathology)]
            blended = (1.0 - a) * e + a * cprompt
            nrm = blended.norm()
            if nrm > 0:
                e = (blended / nrm).float(); n_blend += 1
        qi.query_vector = e.float()
    if ctm_cpu is not None and a > 0.0:
        print(f"  ↳ causal query blend: {n_blend:,} query vectors enriched "
              f"(class_prompt weight={a:.2f})")
    return plans

# ================================================================================
# SECTION 7 — GRAD-CAM
# ================================================================================

class ViTGradCAM:
    def __init__(self, model, preprocess, device, ctm, logit_scale):
        self.model = model; self.preprocess = preprocess; self.device = device
        self.ctm = ctm; self.logit_scale = logit_scale
        self._acts = self._grads = None; self._active = False
        blk = model.visual.transformer.resblocks[-1]
        self._fh = blk.ln_1.register_forward_hook(self._sa)
        self._bh = blk.ln_1.register_full_backward_hook(self._sg)

    def _sa(self, _m, _i, o):
        if self._active:
            self._acts = o

    def _sg(self, _m, _gi, go):
        if self._active:
            self._grads = go[0]

    def _pt(self, x):
        if x is None or x.dim() != 3:
            return None
        if x.shape[0] >= 50:
            return x[1:, 0, :]
        elif x.shape[1] >= 50:
            return x[0, 1:, :]
        return None

    def _cam(self):
        a = self._pt(self._acts); g = self._pt(self._grads)
        if a is None or g is None:
            return None
        a = a.float(); g = g.float()
        c = F.relu((a * g.mean(0).unsqueeze(0)).sum(-1))
        if c.max() <= 1e-8:
            return None
        c = c / c.max(); s = int(math.sqrt(c.numel()))
        if s * s != c.numel():
            return None
        return F.interpolate(c.reshape(s, s).unsqueeze(0).unsqueeze(0),
            size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False).squeeze().detach()

    @torch.enable_grad()
    def _fb(self, pil, bfn):
        self._acts = self._grads = None; self._active = True
        t = self.preprocess(pil).unsqueeze(0).to(self.device)
        self.model.zero_grad(set_to_none=True)
        f = self.model.encode_image(t).float(); f = f / f.norm(dim=-1, keepdim=True)
        bfn(f).backward(); self._active = False
        return self._cam()

    def compute_combined(self, pil, qv, tidx):
        qv_d = qv.to(self.device).float(); qv_d = qv_d / qv_d.norm()
        sc = self._fb(pil, lambda f: (f * qv_d.unsqueeze(0)).sum(-1).squeeze())
        cc = self._fb(pil, lambda f: (self.logit_scale.float() * (f @ self.ctm.float().T)).softmax(-1)[0, tidx])
        vs = sc is not None and sc.max().item() > 1e-6
        vc = cc is not None and cc.max().item() > 1e-6
        if vs and vc:
            r = GRADCAM_SNIPPET_W * (sc / sc.max()) + GRADCAM_CLASS_W * (cc / cc.max())
            return r / r.max() if r.max().item() >= 0.05 else None
        return (sc / sc.max()) if vs else ((cc / cc.max()) if vc else None)

    def remove_hooks(self):
        self._fh.remove(); self._bh.remove()

# ================================================================================
# SECTION 8 — GPU PATCH UTILITIES
# ================================================================================

def extract_candidate_boxes_gpu(hmap, scale, stride, thresh):
    p = F.avg_pool2d(hmap.unsqueeze(0).unsqueeze(0).float(),
                     kernel_size=scale, stride=stride, padding=0).squeeze()
    v = (p >= thresh).nonzero(as_tuple=False)
    if v.shape[0] == 0:
        return []
    y1s = (v[:, 0] * stride).cpu().tolist()
    x1s = (v[:, 1] * stride).cpu().tolist()
    return [(int(x), int(y), int(x) + scale, int(y) + scale) for x, y in zip(x1s, y1s)]

def deduplicate_boxes(boxes):
    seen, u = set(), []
    for b in boxes:
        k = tuple(map(int, b))
        if k not in seen:
            seen.add(k); u.append(b)
    return u

def keep_top_boxes_gpu(boxes, hmap, mx):
    if len(boxes) <= mx:
        return boxes
    sc = boxes[0][2] - boxes[0][0]
    p = F.avg_pool2d(hmap.unsqueeze(0).unsqueeze(0).float(),
                     kernel_size=sc, stride=1, padding=0).squeeze()
    pH, pW = p.shape
    scores = [float(p[min(b[1], pH-1), min(b[0], pW-1)].item()) for b in boxes]
    idx = torch.tensor(scores).topk(mx).indices.tolist()
    return [boxes[i] for i in sorted(idx)]

def combined_score_gpu(sem, prob, gc):
    eps = 1e-6
    s01 = ((sem + 1.0) * 0.5).clamp(eps, 1.0)
    return (s01 ** SCORE_W_SEM * prob.clamp(eps, 1.0) ** SCORE_W_PROB *
            gc.clamp(eps, 1.0) ** SCORE_W_GC).float()

@torch.no_grad()
def encode_patch_batch_gpu(crops, model, preprocess, device, bs=PATCH_ENCODE_BATCH_SZ):
    if not crops:
        return torch.zeros(0, 512, device=device)
    outs = []
    for i in range(0, len(crops), bs):
        batch = torch.stack([preprocess(c) for c in crops[i:i+bs]]).to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=True):
            f = model.encode_image(batch).float(); f = f / f.norm(dim=-1, keepdim=True)
        outs.append(f)
    return torch.cat(outs)

@torch.no_grad()
def zeroshot_probs_gpu(embs, ctm, ls):
    return (ls.float() * (embs.float() @ ctm.float().T)).softmax(-1)

def _gc_scores_batch(boxes, gcam_gpu, device):
    if gcam_gpu is None:
        return torch.full((len(boxes),), 0.5, device=device)
    scores = torch.empty(len(boxes), device=device)
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        scores[i] = gcam_gpu[y1:y2, x1:x2].mean()
    return scores

def _causal_contains(box, cx_p, cy_p, mask, peaks):
    """[C-FIX-1] Box↔blob containment. Returns True if the box legitimately sits on
       the persistence blob. Modes: center | peak | cover | peak_or_cover."""
    x1, y1, x2, y2 = box
    center_in = bool(mask[min(cy_p, IMAGE_SIZE-1), min(cx_p, IMAGE_SIZE-1)].item() > 0.5)
    if CAUSAL_CONTAIN_MODE == "center":
        return center_in
    peak_in = any(x1 <= px < x2 and y1 <= py < y2 for (py, px) in peaks) if peaks else False
    if CAUSAL_CONTAIN_MODE == "peak":
        return center_in or peak_in
    sub = mask[y1:y2, x1:x2]
    cover = (float(sub.sum().item()) / max(float(mask.sum().item()), 1.0)) if sub.numel() else 0.0
    if CAUSAL_CONTAIN_MODE == "cover":
        return center_in or cover >= CAUSAL_MASK_COVER_FRAC
    return center_in or peak_in or cover >= CAUSAL_MASK_COVER_FRAC   # peak_or_cover

def classify_patches_gpu(boxes, embs_gpu, probs_gpu, gcam_gpu,
                          qi, target_idx, zoom_level, image_name, device,
                          persistent_mask=None, persistent_peaks=None):
    """Label each in-anatomy candidate causal or not. Returns (causal, all_scored)."""
    if not boxes or qi.query_vector is None:
        return [], []
    qv = qi.query_vector.to(device).float(); qv = qv / qv.norm()
    sims = (embs_gpu.float() * qv.unsqueeze(0)).sum(-1)
    cls_prob = probs_gpu[:, target_idx]
    gc_vals = _gc_scores_batch(boxes, gcam_gpu, device)
    combo = combined_score_gpu(sims, cls_prob, gc_vals)

    sem_thr = SEMANTIC_THRESHOLD_PER_CLASS.get(qi.pathology, SEMANTIC_THRESHOLD)
    region = qi.anatomical_prior.region_name if qi.anatomical_prior else "unknown"
    peaks = persistent_peaks or []

    sims_cpu = sims.cpu().numpy(); probs_cpu = cls_prob.cpu().numpy()
    gc_cpu = gc_vals.cpu().numpy(); combo_cpu = combo.cpu().numpy()

    causal, all_scored = [], []
    for i, box in enumerate(boxes):
        sim = float(sims_cpu[i]); prob = float(probs_cpu[i])
        gc = float(gc_cpu[i]); cv = float(combo_cpu[i])
        cx_p = (box[0] + box[2]) // 2; cy_p = (box[1] + box[3]) // 2
        has_sem = sim >= sem_thr

        # CAUSAL: semantic signal AND box OVERLAPS the persistence blob
        # (peak-in or coverage — was center-pixel-in-mask, which starved causal
        #  of large boxes and pushed 65% to fallback)
        if persistent_mask is not None:
            is_causal = has_sem and _causal_contains(box, cx_p, cy_p, persistent_mask, peaks)
        else:
            is_causal = has_sem and gc >= GRADCAM_WEAK_THR

        doc = PatchDocument(
            image_name=image_name, pathology=qi.pathology, scale=int(box[2] - box[0]),
            box=tuple(map(int, box)), visual_embedding=embs_gpu[i].detach().cpu().float(),
            semantic_score=sim, zeroshot_prob=prob, gradcam_score=gc, combined_score=cv,
            causal=is_causal, anatomical_region=region, confidence=float(qi.confidence),
            text_snippet=str(qi.text_snippet), zoom_level=zoom_level,
            spurious_source="in_anatomy", selection_source="threshold")
        all_scored.append(doc)
        if is_causal:
            causal.append(doc)
    return causal, all_scored

def select_spurious_in_anatomy(all_scored, sem_thr, max_keep=SPUR_IN_MAX_KEEP):
    """[M-2] Non-causal in-anatomy boxes, lowest-sim, with structure floor + net."""
    noncausal = [d for d in all_scored if not d.causal]
    if not noncausal:
        return []
    seen, nc = set(), []
    for d in sorted(noncausal, key=lambda x: x.semantic_score):
        if d.box not in seen:
            seen.add(d.box); nc.append(d)
    gated = [d for d in nc if d.semantic_score < sem_thr and d.gradcam_score >= SPUR_IN_GC_FLOOR]
    if not gated:
        gated = nc[:max_keep]
        _COUNTERS["spur_in_safetynet"] += 1
    spin = gated[:max_keep]
    for d in spin:
        d.causal = False
        d.spurious_source = "in_anatomy"
        d.selection_source = "threshold"
    return spin

# ================================================================================
# SECTION 8b — PERSISTENT FILTRATION (returns mask + peaks; dilated)
# ================================================================================

def _persistent_mask_scipy(gcam_np, top_k=PERSISTENCE_TOP_K, n_levels=PERSISTENCE_N_LEVELS):
    """Returns (mask, peaks). peaks is a list of (py, px) blob peak pixels."""
    H, W = gcam_np.shape
    vmax = float(gcam_np.max()); vmin = float(max(gcam_np.min(), 0.0))
    if vmax - vmin < 0.01:
        return None, []
    thresholds = np.linspace(vmax * 0.98, vmin + 1e-4, n_levels)
    alive = {}; dead = []; nxt = 0
    prev_labels = np.zeros((H, W), dtype=np.int32); prev_map = {}
    for t in thresholds:
        binary = (gcam_np >= t); labels, n = ndimage_label(binary)
        curr_map = {}
        for c in range(1, n + 1):
            cmask = (labels == c); overlap = prev_labels[cmask]
            parents = set()
            for pl in np.unique(overlap):
                if pl > 0 and pl in prev_map:
                    parents.add(prev_map[pl])
            if len(parents) == 0:
                vals = gcam_np.copy(); vals[~cmask] = -1
                peak = np.unravel_index(vals.argmax(), (H, W))
                alive[nxt] = {'birth': float(t), 'peak': peak}; curr_map[c] = nxt; nxt += 1
            elif len(parents) == 1:
                curr_map[c] = list(parents)[0]
            else:
                sp = sorted(parents, key=lambda i: alive[i]['birth'], reverse=True)
                elder = sp[0]
                for younger in sp[1:]:
                    if younger in alive:
                        dead.append((alive[younger]['birth'] - t,
                                     alive[younger]['birth'], alive[younger]['peak']))
                        del alive[younger]
                curr_map[c] = elder
        prev_labels = labels; prev_map = curr_map
    for cid, info in alive.items():
        dead.append((info['birth'] - vmin, info['birth'], info['peak']))
    if not dead:
        return None, []
    dead.sort(key=lambda x: -x[0])
    mask = np.zeros((H, W), dtype=bool)
    peaks = []
    for k in range(min(top_k, len(dead))):
        pers, birth, peak_yx = dead[k]
        if pers < 0.01:
            continue
        candidate = (gcam_np >= birth * 0.95)
        labels, n = ndimage_label(candidate)
        if n == 0:
            continue
        py, px = peak_yx; cid = labels[py, px]
        if cid > 0:
            comp = (labels == cid)
            if comp.sum() >= PERSISTENCE_MIN_AREA:
                mask |= comp
                peaks.append((int(py), int(px)))
    if mask.sum() < PERSISTENCE_MIN_AREA:
        return None, []
    return mask, peaks

def compute_persistent_causal_mask(gcam_gpu):
    """Returns (mask_tensor_or_None, peaks_list). Blob is dilated by CAUSAL_MASK_DILATE_PX."""
    if gcam_gpu is None or not PERSISTENCE_ENABLED:
        return None, []
    gcam_np = gcam_gpu.cpu().numpy().astype(np.float64)
    if gcam_np.max() - gcam_np.min() < 0.01:
        return None, []
    mask, peaks = _persistent_mask_scipy(gcam_np)
    if mask is None:
        return None, []
    if CAUSAL_MASK_DILATE_PX > 0:                      # [C-FIX-2]
        mask = binary_dilation(mask, iterations=int(CAUSAL_MASK_DILATE_PX))
    _COUNTERS["persistence_used"] += 1
    return torch.from_numpy(mask.astype(np.float32)).to(gcam_gpu.device), peaks

# ================================================================================
# SECTION 8c — OUTSIDE-ANATOMY SCAN  [M-3] (UNCHANGED)
# ================================================================================

def scan_all_outside_anatomy(pil_img, plan, model, preprocess, device,
                             ctm_gpu, ls_gpu, composite_hmap):
    inv_hmap = (1.0 - composite_hmap.clamp(0, 1)).clamp(0, 1)
    qvs = [qi.query_vector for qi in plan.query_items
           if qi.query_vector is not None and not qi.negated]
    qmat = (torch.stack([q.to(device).float() / q.to(device).float().norm() for q in qvs])
            if qvs else None)
    gray = np.asarray(pil_img.convert("L"), dtype=np.float32) / 255.0

    def _score_boxes(boxes):
        if not boxes:
            return []
        crops = [pil_img.crop(b) for b in boxes]
        embs = encode_patch_batch_gpu(crops, model, preprocess, device)
        if qmat is not None:
            sims = (embs.float() @ qmat.T).max(dim=1).values
        else:
            sims = torch.zeros(embs.shape[0], device=device)
        sims_cpu = sims.cpu().numpy()
        out = []
        for i, b in enumerate(boxes):
            x1, y1, x2, y2 = b; patch = gray[y1:y2, x1:x2]
            out.append({"box": b, "emb": embs[i].detach().cpu().float(),
                        "sim": float(sims_cpu[i]),
                        "mean": float(patch.mean()) if patch.size else 0.0,
                        "std": float(patch.std()) if patch.size else 0.0})
        return out

    geo_boxes = []
    for sc in OUTSIDE_ANAT_SCALES:
        stride = max(16, STRIDE_BASE * sc // 128)
        geo_boxes += extract_candidate_boxes_gpu(inv_hmap, sc, stride, OUTSIDE_ANAT_INV_THRESH)
    geo_boxes = deduplicate_boxes(geo_boxes)[:OUTSIDE_ANAT_MAX_CAND]

    border = torch.zeros(IMAGE_SIZE, IMAGE_SIZE, device=device)
    bw = int(IMAGE_SIZE * ARTIFACT_BORDER_FRAC)
    border[:bw, :] = 1.0; border[-bw:, :] = 1.0; border[:, :bw] = 1.0; border[:, -bw:] = 1.0
    ap = (border * inv_hmap).clamp(0, 1)
    int_boxes = []
    for sc in ARTIFACT_SCALES:
        stride = max(16, STRIDE_BASE * sc // 128)
        int_boxes += extract_candidate_boxes_gpu(ap, sc, stride, 0.35)
    int_boxes = deduplicate_boxes(int_boxes)[:OUTSIDE_ANAT_MAX_CAND]

    geo_scored = _score_boxes(geo_boxes)
    int_scored = _score_boxes(int_boxes)

    def _select(scored, sim_cap, max_keep, sel_src):
        if not scored:
            return []
        gated = [s for s in scored if s["sim"] < sim_cap]
        if not gated:
            gated = sorted(scored, key=lambda s: s["sim"])[:max_keep]
            _COUNTERS[f"outside_{sel_src}_safetynet"] += 1
        gated = sorted(gated, key=lambda s: s["std"], reverse=True)[:max_keep]
        docs = []
        for s in gated:
            b = s["box"]
            docs.append(PatchDocument(
                image_name=plan.image_name, pathology="domain_artifact",
                scale=int(b[2] - b[0]), box=tuple(map(int, b)),
                visual_embedding=s["emb"],
                semantic_score=s["sim"], zeroshot_prob=0.0,
                gradcam_score=s["std"], combined_score=s["mean"],
                causal=False, anatomical_region="outside_anatomy", confidence=1.0,
                text_snippet=f"{sel_src}_m={s['mean']:.3f}_s={s['std']:.3f}",
                zoom_level=1, spurious_source="outside_anatomy", selection_source=sel_src))
        return docs

    geo_docs = _select(geo_scored, OUTSIDE_ANAT_SIM_CAP, OUTSIDE_ANAT_MAX_KEEP, "geometric_scan")
    int_docs = _select(int_scored, ARTIFACT_SIM_CAP, ARTIFACT_MAX_KEEP, "intensity_scan")

    seen = set(d.box for d in geo_docs); merged = list(geo_docs)
    for d in int_docs:
        if d.box not in seen:
            seen.add(d.box); merged.append(d)
    if merged:
        _COUNTERS["outside_anatomy_patches"] += len(merged)
        _COUNTERS["images_with_outside_anatomy"] += 1
    return merged[:OUTSIDE_ANAT_MAX_KEEP + ARTIFACT_MAX_KEEP]

# ================================================================================
# SECTION 9 — STAGE 3 PATCH DISCOVERY (resumable + time-budgeted)
# ================================================================================

def discover_patches_for_plan(plan, gradcam, model, preprocess, device, ctm, ls, split_name=""):
    result = ImagePatchResult(image_name=plan.image_name, image_path=plan.image_path, split=split_name)
    if not plan.image_path or not os.path.isfile(plan.image_path):
        _COUNTERS["images_missing"] += 1; return result
    if not plan.query_items:
        return result
    try:
        pil = Image.open(plan.image_path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
    except Exception:
        _COUNTERS["images_failed"] += 1; return result

    cmap = {c.lower(): c for c in TARGET_CLASSES}
    comp_hmap = get_composite_heatmap_gpu(plan, IMAGE_SIZE, IMAGE_SIZE, device)
    rc, rs = [], []

    for qi in plan.query_items:
        if qi.negated or qi.query_vector is None:
            continue
        key = qi.pathology.lower().strip()
        if key not in cmap:
            continue
        qi.pathology = cmap[key]; tidx = TARGET_CLASSES.index(qi.pathology)
        sem_thr = SEMANTIC_THRESHOLD_PER_CLASS.get(qi.pathology, SEMANTIC_THRESHOLD)

        if qi.anatomical_prior:
            p = qi.anatomical_prior
            ihmap = make_gaussian_heatmap_gpu(p.center_x, p.center_y, p.sigma_x, p.sigma_y,
                                              IMAGE_SIZE, IMAGE_SIZE, device)
        else:
            ihmap = comp_hmap.clone()
        mx = ihmap.max()
        if mx > 0:
            ihmap = ihmap / mx

        gcam = gradcam.compute_combined(pil, qi.query_vector, tidx) if ENABLE_GRADCAM else None
        pmask, ppeaks = compute_persistent_causal_mask(gcam)      # [C-FIX] now returns peaks
        if pmask is None:
            _COUNTERS["persistence_fallback"] += 1

        ac, asc = [], []
        scales = _SCALE_MAP.get(qi.pathology, PATCH_SCALES)

        for it in range(1, MAX_ITER + 1):
            sp = SPATIAL_THRESH * (0.70 ** (it - 1)); ih = ihmap.clone()
            if it > 1:
                ih = (ih * 1.30).clamp(0, 1); result.refined = True
            for sc in scales:
                stride = max(16, STRIDE_BASE * sc // 128)
                boxes = keep_top_boxes_gpu(deduplicate_boxes(
                    extract_candidate_boxes_gpu(ih, sc, stride, sp)), ih, MAX_CANDIDATE_BOXES)
                if not boxes:
                    continue
                embs = encode_patch_batch_gpu([pil.crop(b) for b in boxes], model, preprocess, device)
                probs = zeroshot_probs_gpu(embs, ctm, ls)
                c, a = classify_patches_gpu(boxes, embs, probs, gcam, qi, tidx, it,
                                            plan.image_name, device,
                                            persistent_mask=pmask, persistent_peaks=ppeaks)
                ac.extend(c); asc.extend(a)
            if len(ac) >= MIN_CAUSAL_PATCHES:
                if max((d.zeroshot_prob for d in ac), default=0) >= CONF_THRESHOLD or it == MAX_ITER:
                    break

        if 0 < len(ac) < MIN_CAUSAL_PATCHES and ENABLE_ZOOM_REFINEMENT:
            top = sorted(ac, key=lambda d: d.combined_score, reverse=True)[0]
            x1, y1, x2, y2 = top.box; cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
            w, h = x2 - x1, y2 - y1
            ex1 = max(0, int(cx - w * 0.75)); ey1 = max(0, int(cy - h * 0.75))
            ex2 = min(IMAGE_SIZE, int(cx + w * 0.75)); ey2 = min(IMAGE_SIZE, int(cy + h * 0.75))
            if ex2 > ex1 and ey2 > ey1:
                zpil = pil.crop((ex1, ey1, ex2, ey2)).resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
                sxr = (ex2 - ex1) / IMAGE_SIZE; syr = (ey2 - ey1) / IMAGE_SIZE
                zg = gradcam.compute_combined(zpil, qi.query_vector, tidx) if ENABLE_GRADCAM else None
                zpm, zpeaks = compute_persistent_causal_mask(zg)
                zhmap = torch.ones(IMAGE_SIZE, IMAGE_SIZE, device=device)
                for sc in ZOOM_SCALES:
                    stride = max(12, STRIDE_BASE * sc // 128)
                    zb = deduplicate_boxes(extract_candidate_boxes_gpu(
                        zhmap, sc, stride, 0.0))[:MAX_ZOOM_CANDIDATE_BOXES]
                    if not zb:
                        continue
                    ze = encode_patch_batch_gpu([zpil.crop(b) for b in zb], model, preprocess, device)
                    zp = zeroshot_probs_gpu(ze, ctm, ls)
                    zc, za = classify_patches_gpu(zb, ze, zp, zg, qi, tidx, 2,
                                                  plan.image_name, device,
                                                  persistent_mask=zpm, persistent_peaks=zpeaks)
                    for d in za:
                        bx = d.box
                        d.box = (max(0, min(IMAGE_SIZE, int(ex1 + bx[0] * sxr))),
                                 max(0, min(IMAGE_SIZE, int(ey1 + bx[1] * syr))),
                                 max(0, min(IMAGE_SIZE, int(ex1 + bx[2] * sxr))),
                                 max(0, min(IMAGE_SIZE, int(ey1 + bx[3] * syr))))
                    ac.extend(zc); asc.extend(za)
                result.refined = True

        if not ac and asc:
            pool = [d for d in asc if d.semantic_score >= sem_thr * 0.6]
            if pool:
                fb = sorted(pool, key=lambda d: d.combined_score, reverse=True)[:FALLBACK_TOP_K]
                for d in fb:
                    d.causal = True; d.selection_source = "fallback"
                ac.extend(fb); result.used_fallback = True

        spin = select_spurious_in_anatomy(asc, sem_thr, max_keep=SPUR_IN_MAX_KEEP)
        rs.extend(spin)

        rc.extend(sorted(ac, key=lambda d: d.combined_score, reverse=True)[:TOP_K_PER_FIND])

    rs.extend(scan_all_outside_anatomy(pil, plan, model, preprocess, device, ctm, ls, comp_hmap))

    result.causal_patches = rc; result.spurious_patches = rs
    result.n_iterations = MAX_ITER if result.refined else 1
    return result

def run_stage3(plans, split_name, gradcam, model, preprocess, device, ctm, ls, deadline=None):
    ckpt = f"{OUT_DIR}/stage3_{split_name}_ckpt.pkl"
    start = 0; results = []
    if os.path.exists(ckpt):
        with open(ckpt, "rb") as f:
            results = pickle.load(f)
        start = len(results)
        print(f"  ♻️ Resuming {split_name} from checkpoint: {start:,}/{len(plans):,}")
    n_fail = 0; n_empty = sum(1 for r in results if not r.causal_patches)
    t0 = time.time(); last_ckpt_t = t0; stopped_early = False

    pbar = tqdm(range(start, len(plans)), desc=f"Stage 3 · {split_name}",
                initial=start, total=len(plans))
    for idx in pbar:
        if deadline is not None and time.time() > deadline:
            _atomic_pickle(results, ckpt)
            stopped_early = True
            print(f"\n⏰ [{split_name}] Wall-clock budget reached at {idx:,}/{len(plans):,}. "
                  f"Checkpoint saved — re-run the cell to resume.")
            break
        try:
            res = discover_patches_for_plan(plans[idx], gradcam, model, preprocess,
                                            device, ctm, ls, split_name)
        except Exception as e:
            print(f"⚠️ {plans[idx].image_name}: {e}")
            res = ImagePatchResult(image_name=plans[idx].image_name,
                                   image_path=plans[idx].image_path, split=split_name)
            n_fail += 1
        results.append(res)
        if not res.causal_patches:
            n_empty += 1

        now = time.time()
        done_now = len(results) - start
        if (done_now % CKPT_EVERY_IMAGES == 0) or ((now - last_ckpt_t) > CKPT_EVERY_MIN * 60):
            torch.cuda.empty_cache(); gc.collect()
            _atomic_pickle(results, ckpt)
            last_ckpt_t = now
            rate = done_now / max(now - t0, 1e-6)
            remaining = len(plans) - len(results)
            eta_h = remaining / max(rate, 1e-9) / 3600.0
            pbar.set_postfix_str(f"{rate*3600:,.0f} img/h | ETA {eta_h:.2f}h | "
                                 f"budget {_budget_left_h():.2f}h")

    _atomic_pickle(results, ckpt)
    completed = (len(results) >= len(plans))
    if completed:
        print(f"  {split_name}: {n_fail} failed | {n_empty:,} empty | "
              f"{len(results)-n_empty:,} w/causal | ✅ complete")
        if os.path.exists(ckpt):
            os.remove(ckpt)
    else:
        print(f"  {split_name}: partial {len(results):,}/{len(plans):,} "
              f"({len(results)-n_empty:,} w/causal). Checkpoint kept for resume.")
    return results, completed, stopped_early

# ================================================================================
# SECTION 10 — VISUALIZATION
# ================================================================================

_VC = {("causal", "threshold"): (0, 220, 60, 90), ("causal", "fallback"): (30, 140, 255, 90),
       ("spurious", "in_anatomy"): (220, 30, 30, 70), ("spurious", "outside_anatomy"): (255, 160, 0, 70)}
_VO = {("causal", "threshold"): (0, 200, 50, 255), ("causal", "fallback"): (20, 120, 240, 255),
       ("spurious", "in_anatomy"): (200, 20, 20, 255), ("spurious", "outside_anatomy"): (230, 140, 0, 255)}

def save_patch_overlay(result, split_name):
    if not ENABLE_VISUALIZATION or not result.image_path or not os.path.isfile(result.image_path):
        return
    try:
        img = Image.open(result.image_path).convert("RGBA").resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
        ov = Image.new("RGBA", img.size, (0, 0, 0, 0)); dr = ImageDraw.Draw(ov)

        def db(p, pt):
            k = (("causal", "fallback" if p.selection_source == "fallback" else "threshold")
                 if pt == "causal" else ("spurious", p.spurious_source))
            x1, y1, x2, y2 = p.box
            dr.rectangle([x1, y1, x2, y2], fill=_VC.get(k, (128, 128, 128, 60)))
            dr.rectangle([x1, y1, x2, y2], outline=_VO.get(k, (128, 128, 128, 255))[:3], width=2)
            dr.text((x1 + 2, y1 + 2), f"{p.pathology[:3]} {p.combined_score:.2f}", fill=(255, 255, 255, 230))

        for p in result.causal_patches:
            db(p, "causal")
        for p in result.spurious_patches:
            db(p, "spurious")
        comp = Image.alpha_composite(img, ov).convert("RGB")
        leg = Image.new("RGB", (IMAGE_SIZE, 48), (20, 20, 20)); ld = ImageDraw.Draw(leg); xo = 4
        for col, lab in [((0, 220, 60), "causal"), ((30, 140, 255), "fallback"),
                         ((220, 30, 30), "spur-in"), ((255, 160, 0), "spur-out")]:
            ld.rectangle([xo, 8, xo + 12, 40], fill=col)
            ld.text((xo + 15, 14), lab, fill=(200, 200, 200)); xo += len(lab) * 7 + 22
        full = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE + 48))
        full.paste(comp, (0, 0)); full.paste(leg, (0, IMAGE_SIZE))
        full.save(f"{VIS_DIR}/{split_name}/"
                  f"{os.path.splitext(os.path.basename(result.image_name))[0]}_patches.png")
    except Exception:
        _COUNTERS["vis_failed"] += 1

# ================================================================================
# SECTION 11 — SAVE / SUMMARY
# ================================================================================

_MK = ["image_name", "pathology", "box", "scale", "semantic_score", "zeroshot_prob",
       "gradcam_score", "combined_score", "anatomical_region", "confidence",
       "text_snippet", "zoom_level"]

def _md(p, sp, ek):
    d = {"split": sp}
    for k in _MK:
        d[k] = getattr(p, k)
    for k in ek:
        d[k] = getattr(p, k)
    return d

def split_embeddings_three_ways(rd):
    ce, cm = [], []; se, sm = [], []; oe, om = [], []
    for sp, results in rd.items():
        for r in results:
            for p in r.causal_patches:
                ce.append(p.visual_embedding.float()); cm.append(_md(p, sp, ["selection_source"]))
            for p in r.spurious_patches:
                if p.spurious_source == "outside_anatomy":
                    oe.append(p.visual_embedding.float())
                    om.append(_md(p, sp, ["spurious_source", "selection_source"]))
                else:
                    se.append(p.visual_embedding.float())
                    sm.append(_md(p, sp, ["spurious_source", "selection_source"]))

    def pk(e, m):
        return {"embeddings": torch.stack(e) if e else torch.zeros(0, 512), "meta": m, "n": len(e)}
    return pk(ce, cm), pk(se, sm), pk(oe, om)

def summarize_stage3(results, split):
    nc = sum(len(r.causal_patches) for r in results)
    ne = sum(1 for r in results if r.causal_patches)
    nfb = sum(sum(1 for p in r.causal_patches if p.selection_source == "fallback") for r in results)
    nfi = sum(1 for r in results if r.used_fallback)
    n_true = nc - nfb
    nsi = sum(sum(1 for p in r.spurious_patches if p.spurious_source == "in_anatomy") for r in results)
    nso = sum(sum(1 for p in r.spurious_patches if p.spurious_source == "outside_anatomy") for r in results)
    nsi_img = sum(1 for r in results if any(p.spurious_source == "in_anatomy" for p in r.spurious_patches))
    ng = sum(sum(1 for p in r.spurious_patches if p.selection_source == "geometric_scan") for r in results)
    ni = sum(sum(1 for p in r.spurious_patches if p.selection_source == "intensity_scan") for r in results)
    fb_pct = 100.0 * nfb / max(nc, 1)
    print(f"{split:<6}: {len(results):,} imgs | {ne:,} w/causal | "
          f"{nc:,} causal ({n_true} true / {nfb} fb = {fb_pct:.1f}% fb / {nfi} imgs) | "
          f"spur_in={nsi:,} ({nsi_img} imgs) spur_out={nso:,} (geo={ng:,} int={ni:,})")

# ================================================================================
# SECTION 12 — MAIN EXECUTION
# ================================================================================

print("\n" + "="*70 + "\nREADING CSV\n" + "="*70)
train_df = read_pairs_csv(TRAIN_CSV); val_df = read_pairs_csv(VAL_CSV); test_df = read_pairs_csv(TEST_CSV)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

if MULTICLASS_ONLY:
    train_df = filter_multiclass_rows(train_df, "Train")
    val_df   = filter_multiclass_rows(val_df,   "Val")

if SMOKE_TEST:
    train_df = train_df.sample(min(len(train_df), SMOKE_TRAIN_N), random_state=SEED).reset_index(drop=True)
    val_df   = val_df.sample(min(len(val_df),   SMOKE_VAL_N),   random_state=SEED).reset_index(drop=True)
    print(f"🧪 SMOKE_TEST: train={len(train_df)} val={len(val_df)} (speed probe only)")
else:
    if MAX_TRAIN_IMAGES is not None and len(train_df) > MAX_TRAIN_IMAGES:
        train_df = train_df.sample(MAX_TRAIN_IMAGES, random_state=SEED).reset_index(drop=True)
    if MAX_VAL_IMAGES is not None and len(val_df) > MAX_VAL_IMAGES:
        val_df = val_df.sample(MAX_VAL_IMAGES, random_state=SEED).reset_index(drop=True)
        print(f"🎯 Val subset: {len(val_df):,} images (random_state={SEED})")

print(f"➡️  Processing — Train: {len(train_df):,} | Val: {len(val_df):,}")
print(f"⏱️  Wall-clock budget: {WALLCLOCK_BUDGET_HOURS:.1f}h from cell start "
      f"(≈{_budget_left_h():.2f}h left now)")
print(f"🔧 Causal containment: mode={CAUSAL_CONTAIN_MODE} cover={CAUSAL_MASK_COVER_FRAC} "
      f"dilate={CAUSAL_MASK_DILATE_PX}px top_k={PERSISTENCE_TOP_K}")

print("\n" + "="*70 + "\nSTAGE 2 — RULE-BASED QUERY PLANS\n" + "="*70)
VOCAB = mine_vocab_from_reports(train_df)
train_plans = run_stage2(train_df, VOCAB, MAX_TRAIN_IMAGES if not SMOKE_TEST else None,
                         "Stage 2 · train", USE_LABEL_BACKUP_TRAIN)
val_plans   = run_stage2(val_df,   VOCAB, None, "Stage 2 · val", USE_LABEL_BACKUP_VAL)
test_plans  = run_stage2(test_df,  VOCAB, None, "Stage 2 · test", False)
split_stats(train_plans, "Train"); split_stats(val_plans, "Val"); split_stats(test_plans, "Test")

with open(STAGE2_OUT, "wb") as f:
    pickle.dump({"train": train_plans, "val": val_plans, "test": test_plans, "vocab": VOCAB,
                 "metadata": {"version": "v11", "method": "rule_only",
                              "multiclass_only": MULTICLASS_ONLY,
                              "target_classes": TARGET_CLASSES}}, f, protocol=4)
print(f"✅ Stage 2 → {STAGE2_OUT}")

print("\n" + "="*70 + "\nSTAGE 3 — PATCH DISCOVERY (3 buckets)\n" + "="*70)
chexzero, cz_prep = load_chexzero(CHEXZERO_CKPT, Device)
ctm_gpu, ls_gpu = build_class_text_matrix(chexzero, Device)
train_plans = fill_query_vectors(train_plans, chexzero, Device, ctm=ctm_gpu)
val_plans   = fill_query_vectors(val_plans,   chexzero, Device, ctm=ctm_gpu)
gradcam = ViTGradCAM(model=chexzero, preprocess=cz_prep, device=Device, ctm=ctm_gpu, logit_scale=ls_gpu)

train_completed = val_completed = False
train_results, val_results = [], []
try:
    train_results, train_completed, train_stopped = run_stage3(
        train_plans, "train", gradcam, chexzero, cz_prep, Device, ctm_gpu, ls_gpu, deadline=_DEADLINE)
    if train_completed and _budget_left_h() > 0.15:
        val_results, val_completed, val_stopped = run_stage3(
            val_plans, "val", gradcam, chexzero, cz_prep, Device, ctm_gpu, ls_gpu, deadline=_DEADLINE)
    else:
        if not train_completed:
            print("⏭️  Skipping val this session — train not complete. Re-run to resume train, then val.")
        else:
            print("⏭️  Skipping val this session — too little time budget left. Re-run to do val.")
finally:
    gradcam.remove_hooks()

if ENABLE_VISUALIZATION and VIS_MAX_PER_SPLIT > 0:
    for r in tqdm(train_results[:VIS_MAX_PER_SPLIT], desc="Vis·train"):
        save_patch_overlay(r, "train")
    for r in tqdm(val_results[:VIS_MAX_PER_SPLIT], desc="Vis·val"):
        save_patch_overlay(r, "val")

rd = {"train": train_results, "val": val_results}
print("\n" + "="*70 + "\nSTAGE 3 SUMMARY\n" + "="*70)
summarize_stage3(train_results, "Train")
if val_results:
    summarize_stage3(val_results, "Val")
print(f"\nCounters: {dict(_COUNTERS)}")

# --- bucket + fallback health check ---
_all = train_results + val_results
_c    = sum(len(r.causal_patches) for r in _all)
_cfb  = sum(sum(1 for p in r.causal_patches if p.selection_source == "fallback") for r in _all)
_call = sum(1 for r in _all if r.causal_patches and all(p.selection_source == "fallback" for p in r.causal_patches))
_sin  = sum(sum(1 for p in r.spurious_patches if p.spurious_source == "in_anatomy")  for r in _all)
_sout = sum(sum(1 for p in r.spurious_patches if p.spurious_source == "outside_anatomy") for r in _all)
_fbpct = 100.0 * _cfb / max(_c, 1)
print("\n── BUCKET CHECK ──")
print(f"   causal      = {_c:,}  ({_c-_cfb} true / {_cfb} fb = {_fbpct:.1f}% fallback) "
      f"{'✅' if _c > 0 else '⚠️ EMPTY'}")
print(f"   spur_in     = {_sin:,}  {'✅' if _sin > 0 else '⚠️ EMPTY'}")
print(f"   spur_out    = {_sout:,}  {'✅' if _sout> 0 else '⚠️ EMPTY'}")
print(f"   all-fallback images = {_call}  (want this LOW)")
if _fbpct > 30:
    print("   ⚠️ fallback still high — lower CAUSAL_MASK_COVER_FRAC to 0.20 or raise "
          "CAUSAL_MASK_DILATE_PX to 12.")
elif _c > 0 and _fbpct < 3 and _call == 0:
    print("   ⚠️ fallback near-zero — if boxes look oversized, set CAUSAL_CONTAIN_MODE='peak'.")
else:
    print("   ✅ fallback in a healthy range.")

run_complete = bool(train_completed and val_completed)
with open(STAGE3_RESULTS, "wb") as f:
    pickle.dump(rd, f, protocol=4)
cp, sp, op = split_embeddings_three_ways(rd)
torch.save(cp, STAGE3_CAUSAL_PT)
torch.save(sp, STAGE3_SPIN_PT)
torch.save(op, STAGE3_SPOUT_PT)
with open(STAGE3_META, "wb") as f:
    pickle.dump({"version": "v11", "target_classes": TARGET_CLASSES,
                 "complete": run_complete,
                 "train_complete": train_completed, "val_complete": val_completed,
                 "multiclass_only": MULTICLASS_ONLY,
                 "counts": {"causal": cp["n"], "spur_in": sp["n"], "spur_out": op["n"]},
                 "causal_query_class_blend": CAUSAL_QUERY_CLASS_BLEND,
                 "causal_containment": {"mode": CAUSAL_CONTAIN_MODE,
                                        "cover_frac": CAUSAL_MASK_COVER_FRAC,
                                        "dilate_px": CAUSAL_MASK_DILATE_PX},
                 "persistence": {"enabled": PERSISTENCE_ENABLED, "top_k": PERSISTENCE_TOP_K,
                                 "n_levels": PERSISTENCE_N_LEVELS, "backend": "scipy"},
                 "thresholds": {"semantic_per_class": SEMANTIC_THRESHOLD_PER_CLASS,
                                "gradcam_weak_fallback": GRADCAM_WEAK_THR,
                                "spur_in_gc_floor": SPUR_IN_GC_FLOOR,
                                "outside_anat_sim_cap": OUTSIDE_ANAT_SIM_CAP},
                 "counters": dict(_COUNTERS),
                 "fixes": ["v10_all",
                           "C_FIX_1_peak_or_cover_containment",
                           "C_FIX_2_mask_dilation",
                           "C_FIX_3_persistence_top_k_2"]}, f, protocol=4)

print(f"\n{'✅ DONE — v11 (COMPLETE)' if run_complete else '💾 v11 PARTIAL saved — re-run cell to resume'}")
print(f"   Causal  → {STAGE3_CAUSAL_PT}  ({cp['n']:,})")
print(f"   SpurIn  → {STAGE3_SPIN_PT}    ({sp['n']:,})")
print(f"   SpurOut → {STAGE3_SPOUT_PT}   ({op['n']:,})")
print(f"   Meta    → {STAGE3_META}  (complete={run_complete})")
if not run_complete:
    print("   ↻ Re-run this same cell: train resumes from its checkpoint, then val runs.")
print("⚠️  Test FROZEN — Stage 3 on test only at final eval.")

✅ GPU : Tesla T4
✅ VRAM: 15.6 GB

READING CSV
Train: 51,502 | Val: 9,177 | Test: 5,110
🎯 Train: multi-class filter kept 10,877/51,502 (label cols: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion'])
🎯 Val: multi-class filter kept 1,985/9,177 (label cols: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion'])
➡️  Processing — Train: 10,877 | Val: 1,985
⏱️  Wall-clock budget: 9.0h from cell start (≈9.00h left now)
🔧 Causal containment: mode=peak_or_cover cover=0.35 dilate=8px top_k=2

STAGE 2 — RULE-BASED QUERY PLANS

VOCAB MINER: 10,877 reports


Mining vocab: 100%|██████████| 10877/10877 [00:43<00:00, 251.41it/s]


Skipped: 0
  Atelectasis         : 972
  Cardiomegaly        : 628
  Consolidation       : 1,134
  Edema               : 1,630
  Pleural Effusion    : 5,573


Stage 2 · test: 100%|██████████| 5110/5110 [00:16<00:00, 302.04it/s]


Train : 10,877 | 5,344 non-empty | 8,036 items | avg=0.74
Val   : 1,985 | 987 non-empty | 1,520 items | avg=0.77
Test  : 5,110 | 2,278 non-empty | 3,483 items | avg=0.68
✅ Stage 2 → /kaggle/working/stage2_outputs.pkl

STAGE 3 — PATCH DISCOVERY (3 buckets)

LOADING CHEXZERO
Missing: 0 | Unexpected: 0
✅ CheXzero loaded.
✅ CTM: (5, 512) | logit_scale=94.65


Text encode: 100%|██████████| 126/126 [00:05<00:00, 23.03it/s]


  ↳ causal query blend: 8,036 query vectors enriched (class_prompt weight=0.35)


Text encode: 100%|██████████| 24/24 [00:01<00:00, 23.48it/s]


  ↳ causal query blend: 1,520 query vectors enriched (class_prompt weight=0.35)


Stage 3 · train:   4%|▎         | 387/10877 [06:48<1:58:07,  1.48it/s]

# Ablations

In [ ]:
# ================================================================================
# CELL A — PATCH-MINING ABLATIONS + BENCHMARK   (run AFTER the v11 Stage2+3 cell)
# ================================================================================
# Scope: isolates the CANDIDATE→CAUSAL / SPURIOUS patch-mining stage ONLY.
# Reuses the v11 objects already in memory: train_plans, val_plans (query vectors
# filled), chexzero, cz_prep, ctm_gpu, ls_gpu, discover_patches_for_plan,
# get_composite_heatmap_gpu, ViTGradCAM, _persistent_mask_scipy, TARGET_CLASSES,
# train_df, val_df, _find_label_cols. Does NOT touch DDE / prototype / TTDA.
#
# METRIC TIERS
#   TIER-1 (trust most; NO Stage-2 confound — query plans are frozen across configs)
#     true_causal% ...... share of causal patches found by the real gate (not fallback)
#     fallback% ......... complement; the safety-net's share  (want LOW)
#     causal/img ........ mean causal patches per image
#     %img_causal ....... images with >=1 causal patch
#     prior_overlap% .... causal patch centers landing inside the anatomy prior (want HIGH)
#     leak% ............. causal patch centers OUTSIDE anatomy (purity; want ~0)
#     mean_sim .......... mean semantic similarity of causal patches
#     spur_in / spur_out  spurious composition (grid F, descriptive)
#     stability ......... same-seed rerun Jaccard of causal box sets (want ≈1.0)
#   TIER-2 (delta-only; Stage-2-CONFOUNDED absolute value)
#     macroAUROC ........ per-image max causal zeroshot_prob vs GT; ONLY Δ vs FULL is evidence
#
# GRIDS (all on causal-quality knobs — the live issue post v11 fix)
#   A containment mode : center | peak | cover | peak_or_cover(FULL)
#   B persistence Top-K: 1 | 2(FULL) | 3
#   C filtration levels: 16 | 32(FULL) | 48
#   D mask dilation px : 0 | 8(FULL) | 12
#   E coverage frac    : 0.20 | 0.35(FULL) | 0.50
#   F semantic thr ×   : 0.75 | 1.00(FULL) | 1.25
# FULL-equivalent rows are COPIED from the single FULL run (no redundant sweeps).
# ================================================================================

import time, copy, os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

# --------------------------------------------------------------------------------
# A.0 — Ablation config
# --------------------------------------------------------------------------------
ABL_SUBSET_N             = 120     # raise to 300–500 for the FINAL paper table
ABL_SEED                 = 42
ABL_PRIOR_THR            = 0.10    # composite-heatmap value above which a centre is "in-prior"
ABL_WALLCLOCK_BUDGET_MIN = 90      # hard stop between configs (from this cell's start)
ABL_RUN_STABILITY        = True
ABL_STAB_N               = 25
ABL_OUT_CSV              = f"{OUT_DIR}/patch_ablations_v11.csv"

np.random.seed(ABL_SEED)
_ABL_T0 = time.time()
def _abl_left_min():
    return ABL_WALLCLOCK_BUDGET_MIN - (time.time() - _ABL_T0) / 60.0

# --------------------------------------------------------------------------------
# A.1 — Live-tunable persistence override (v11 froze Top-K / N_LEVELS as defaults;
#        this reads the live globals so grids B and C actually take effect)
# --------------------------------------------------------------------------------
def compute_persistent_causal_mask(gcam_gpu):          # overrides the v11 global
    if gcam_gpu is None or not PERSISTENCE_ENABLED:
        return None, []
    gcam_np = gcam_gpu.cpu().numpy().astype(np.float64)
    if gcam_np.max() - gcam_np.min() < 0.01:
        return None, []
    mask, peaks = _persistent_mask_scipy(gcam_np,
                                         top_k=PERSISTENCE_TOP_K,
                                         n_levels=PERSISTENCE_N_LEVELS)
    if mask is None:
        return None, []
    if CAUSAL_MASK_DILATE_PX > 0:
        from scipy.ndimage import binary_dilation
        mask = binary_dilation(mask, iterations=int(CAUSAL_MASK_DILATE_PX))
    return torch.from_numpy(mask.astype(np.float32)).to(gcam_gpu.device), peaks

# --------------------------------------------------------------------------------
# A.2 — Subset (non-empty plans with query vectors) + GT label lookup
# --------------------------------------------------------------------------------
_ALL_PLANS = [p for p in (val_plans + train_plans)
              if p.query_items and any(qi.query_vector is not None for qi in p.query_items)]
rng = np.random.RandomState(ABL_SEED)
if len(_ALL_PLANS) > ABL_SUBSET_N:
    idx = rng.choice(len(_ALL_PLANS), ABL_SUBSET_N, replace=False)
    ABL_PLANS = [_ALL_PLANS[i] for i in sorted(idx)]
else:
    ABL_PLANS = _ALL_PLANS
print(f"Ablation subset: {len(ABL_PLANS)} plans (of {len(_ALL_PLANS)} non-empty)")

def _build_label_lookup():
    lut = {}
    for df in (train_df, val_df):
        lcols = _find_label_cols(df)
        if not lcols:
            continue
        name_col = next((c for c in ["image_name", "Image Index", "image"] if c in df.columns), None)
        if name_col is None:
            continue
        for _, row in df.iterrows():
            nm = os.path.basename(str(row[name_col]))
            vec = {}
            for cls, col in lcols.items():
                try:
                    vec[cls] = 1.0 if float(row[col]) == 1.0 else 0.0
                except Exception:
                    vec[cls] = 0.0
            lut[nm] = vec
    return lut

GT_LUT = _build_label_lookup()
print(f"GT label lookup: {len(GT_LUT):,} images | classes matched: "
      f"{sorted(set().union(*[set(v) for v in list(GT_LUT.values())[:1]])) if GT_LUT else '—'}")

# --------------------------------------------------------------------------------
# A.3 — One evaluation pass over the subset -> full metric dict
# --------------------------------------------------------------------------------
def _eval_current_config(gradcam):
    t0 = time.time()
    n_img = len(ABL_PLANS)
    n_causal = n_true = n_fb = n_img_causal = 0
    n_in_prior = n_leak = 0
    sims = []
    n_spin = n_spout = 0
    # AUROC accumulators
    scores = {c: [] for c in TARGET_CLASSES}
    gts    = {c: [] for c in TARGET_CLASSES}

    for plan in ABL_PLANS:
        res = discover_patches_for_plan(plan, gradcam, chexzero, cz_prep,
                                        Device, ctm_gpu, ls_gpu, "abl")
        cps = res.causal_patches
        if cps:
            n_img_causal += 1
        n_causal += len(cps)
        # prior overlap / leak need the composite heatmap for this plan
        chm = get_composite_heatmap_gpu(plan, IMAGE_SIZE, IMAGE_SIZE, Device)
        for p in cps:
            sims.append(p.semantic_score)
            if p.selection_source == "fallback":
                n_fb += 1
            else:
                n_true += 1
            cx = min((p.box[0] + p.box[2]) // 2, IMAGE_SIZE - 1)
            cy = min((p.box[1] + p.box[3]) // 2, IMAGE_SIZE - 1)
            if float(chm[cy, cx].item()) >= ABL_PRIOR_THR:
                n_in_prior += 1
            else:
                n_leak += 1
        for p in res.spurious_patches:
            if p.spurious_source == "outside_anatomy":
                n_spout += 1
            else:
                n_spin += 1
        # per-image, per-class score = max causal zeroshot_prob (0 if none)
        gt = GT_LUT.get(os.path.basename(plan.image_name))
        if gt is not None:
            best = {c: 0.0 for c in TARGET_CLASSES}
            for p in cps:
                if p.pathology in best:
                    best[p.pathology] = max(best[p.pathology], float(p.zeroshot_prob))
            for c in TARGET_CLASSES:
                scores[c].append(best[c]); gts[c].append(gt.get(c, 0.0))

    # macro AUROC over classes with both GT values present
    aucs = []
    for c in TARGET_CLASSES:
        y = np.array(gts[c]); s = np.array(scores[c])
        if y.sum() > 0 and y.sum() < len(y):
            try:
                aucs.append(roc_auc_score(y, s))
            except Exception:
                pass
    macro_auroc = float(np.mean(aucs)) if aucs else float("nan")

    return {
        "causal_per_img":   n_causal / max(n_img, 1),
        "pct_img_causal":   100.0 * n_img_causal / max(n_img, 1),
        "true_causal_pct":  100.0 * n_true / max(n_causal, 1),
        "fallback_pct":     100.0 * n_fb / max(n_causal, 1),
        "prior_overlap_pct":100.0 * n_in_prior / max(n_causal, 1),
        "leak_pct":         100.0 * n_leak / max(n_causal, 1),
        "mean_sim":         float(np.mean(sims)) if sims else 0.0,
        "spur_in":          n_spin,
        "spur_out":         n_spout,
        "macroAUROC":       macro_auroc,
        "n_auc_classes":    len(aucs),
        "sec_img":          (time.time() - t0) / max(n_img, 1),
    }

# --------------------------------------------------------------------------------
# A.4 — Global-override apply/restore (discover reads these live)
# --------------------------------------------------------------------------------
_ABL_KEYS = ["CAUSAL_CONTAIN_MODE", "CAUSAL_MASK_COVER_FRAC", "CAUSAL_MASK_DILATE_PX",
             "PERSISTENCE_TOP_K", "PERSISTENCE_N_LEVELS", "SEMANTIC_THRESHOLD_PER_CLASS"]
_BASE = {k: copy.deepcopy(globals()[k]) for k in _ABL_KEYS}

def _apply(ov):
    for k in _ABL_KEYS:
        globals()[k] = copy.deepcopy(_BASE[k])
    if "sem_scale" in ov:
        globals()["SEMANTIC_THRESHOLD_PER_CLASS"] = {
            c: v * ov["sem_scale"] for c, v in _BASE["SEMANTIC_THRESHOLD_PER_CLASS"].items()}
    for k, v in ov.items():
        if k == "sem_scale":
            continue
        globals()[k] = v

def _restore():
    for k in _ABL_KEYS:
        globals()[k] = copy.deepcopy(_BASE[k])

# --------------------------------------------------------------------------------
# A.5 — Grid definitions (empty override == FULL, copied not rerun)
# --------------------------------------------------------------------------------
CONFIGS = [
    ("FULL",                      {}),
    # A — containment mode
    ("A:contain=center",          {"CAUSAL_CONTAIN_MODE": "center"}),
    ("A:contain=peak",            {"CAUSAL_CONTAIN_MODE": "peak"}),
    ("A:contain=cover",           {"CAUSAL_CONTAIN_MODE": "cover"}),
    ("A:contain=peak_or_cover",   {}),                                  # == FULL
    # B — persistence Top-K
    ("B:topK=1",                  {"PERSISTENCE_TOP_K": 1}),
    ("B:topK=2",                  {}),                                  # == FULL
    ("B:topK=3",                  {"PERSISTENCE_TOP_K": 3}),
    # C — filtration levels L
    ("C:levels=16",               {"PERSISTENCE_N_LEVELS": 16}),
    ("C:levels=32",               {}),                                  # == FULL
    ("C:levels=48",               {"PERSISTENCE_N_LEVELS": 48}),
    # D — mask dilation
    ("D:dilate=0",                {"CAUSAL_MASK_DILATE_PX": 0}),
    ("D:dilate=8",                {}),                                  # == FULL
    ("D:dilate=12",               {"CAUSAL_MASK_DILATE_PX": 12}),
    # E — coverage fraction
    ("E:cover=0.20",              {"CAUSAL_MASK_COVER_FRAC": 0.20}),
    ("E:cover=0.35",              {}),                                  # == FULL
    ("E:cover=0.50",              {"CAUSAL_MASK_COVER_FRAC": 0.50}),
    # F — semantic threshold scale
    ("F:sem×0.75",                {"sem_scale": 0.75}),
    ("F:sem×1.00",                {}),                                  # == FULL
    ("F:sem×1.25",                {"sem_scale": 1.25}),
]
_ORDER = [lab for lab, _ in CONFIGS]

# --------------------------------------------------------------------------------
# A.6 — Resume-from-CSV + incremental atomic save
# --------------------------------------------------------------------------------
ROWS, done = [], set()
if os.path.exists(ABL_OUT_CSV):
    prev = pd.read_csv(ABL_OUT_CSV)
    ROWS = prev.to_dict("records"); done = set(prev["label"].astype(str))
    print(f"♻️ Resuming: {len(done)} configs already in {ABL_OUT_CSV}")

def _save():
    df = pd.DataFrame(ROWS)
    tmp = ABL_OUT_CSV + ".tmp"; df.to_csv(tmp, index=False); os.replace(tmp, ABL_OUT_CSV)

# --------------------------------------------------------------------------------
# A.7 — Run FULL first (baseline for copies + Δ), then the rest
# --------------------------------------------------------------------------------
gradcam = ViTGradCAM(model=chexzero, preprocess=cz_prep, device=Device,
                     ctm=ctm_gpu, logit_scale=ls_gpu)
row_full = None
try:
    if "FULL" in done:
        row_full = next(r for r in ROWS if str(r["label"]) == "FULL")
        print("FULL loaded from CSV.")
    else:
        _apply({}); row_full = {"label": "FULL", **_eval_current_config(gradcam)}
        ROWS.append(row_full); done.add("FULL"); _save()
        print(f"FULL  macroAUROC={row_full['macroAUROC']:.4f} | "
              f"true_causal%={row_full['true_causal_pct']:.1f} "
              f"fb%={row_full['fallback_pct']:.1f} | {row_full['sec_img']*1000:.0f} ms/img")
    FULL_AUROC = float(row_full["macroAUROC"])
    est_min = row_full["sec_img"] * len(ABL_PLANS) * \
              sum(1 for lab, ov in CONFIGS if ov and lab not in done) / 60.0
    print(f"⏱️ Est. remaining sweep ≈ {est_min:.1f} min (budget {ABL_WALLCLOCK_BUDGET_MIN} min).")

    _stopped = False
    for lab, ov in CONFIGS:
        if lab in done:
            continue
        if not ov:                                     # FULL-equivalent → copy
            r = dict(row_full); r["label"] = lab
            ROWS.append(r); done.add(lab); _save(); continue
        if _abl_left_min() <= 0:
            print(f"\n⏰ Budget reached before '{lab}'. {len(done)} configs saved; re-run to finish.")
            _stopped = True; break
        _apply(ov)
        try:
            r = {"label": lab, **_eval_current_config(gradcam)}
        finally:
            _restore()
        d = r["macroAUROC"] - FULL_AUROC
        print(f"  {lab:<22} AUROC={r['macroAUROC']:.4f} (Δ{d:+.4f}) | "
              f"true%={r['true_causal_pct']:5.1f} fb%={r['fallback_pct']:5.1f} "
              f"prior%={r['prior_overlap_pct']:5.1f} leak%={r['leak_pct']:4.1f} | "
              f"{r['sec_img']*1000:.0f} ms/img | {_abl_left_min():.0f} min left")
        ROWS.append(r); done.add(lab); _save()
finally:
    gradcam.remove_hooks(); _restore()

# --------------------------------------------------------------------------------
# A.8 — Master benchmark table
# --------------------------------------------------------------------------------
_pos = {lab: i for i, lab in enumerate(_ORDER)}
ROWS.sort(key=lambda r: _pos.get(str(r["label"]), 999))
print("\n" + "=" * 108)
print(f"{'config':<22}{'AUROC':>8}{'Δvsfull':>9}{'true%':>7}{'fb%':>6}{'caus/img':>9}"
      f"{'%img':>6}{'prior%':>8}{'leak%':>7}{'sim':>6}{'sp_in':>7}{'sp_out':>7}{'ms/img':>8}")
print("=" * 108)
for r in ROWS:
    d = float(r["macroAUROC"]) - FULL_AUROC
    print(f"{str(r['label']):<22}{float(r['macroAUROC']):>8.4f}{d:>+9.4f}"
          f"{float(r['true_causal_pct']):>7.1f}{float(r['fallback_pct']):>6.1f}"
          f"{float(r['causal_per_img']):>9.2f}{float(r['pct_img_causal']):>6.1f}"
          f"{float(r['prior_overlap_pct']):>8.1f}{float(r['leak_pct']):>7.1f}"
          f"{float(r['mean_sim']):>6.3f}{int(r['spur_in']):>7}{int(r['spur_out']):>7}"
          f"{float(r['sec_img'])*1000:>8.0f}")
print("=" * 108)
print("Trust the Δ column + tier-1 structural cols (true%, fb%, prior%, leak%) for claims.")
print("macroAUROC absolute is Stage-2-confounded (frozen plans); only Δ vs FULL is evidence.")

# --------------------------------------------------------------------------------
# A.9 — Stability (same-seed rerun Jaccard of causal box sets, at FULL settings)
# --------------------------------------------------------------------------------
if ABL_RUN_STABILITY and not _stopped and _abl_left_min() > 2:
    print("\n" + "=" * 70 + f"\nSTABILITY — same-seed Jaccard ({ABL_STAB_N} imgs)\n" + "=" * 70)
    _apply({}); sub = ABL_PLANS[:ABL_STAB_N]
    def _boxsets(plans):
        g = ViTGradCAM(chexzero, cz_prep, Device, ctm_gpu, ls_gpu)
        try:
            out = {}
            for p in plans:
                r = discover_patches_for_plan(p, g, chexzero, cz_prep, Device, ctm_gpu, ls_gpu, "st")
                out[p.image_name] = {tuple(d.box) for d in r.causal_patches}
            return out
        finally:
            g.remove_hooks()
    A = _boxsets(sub); B = _boxsets(sub); _restore()
    js = []
    for k in A:
        u = A[k] | B[k]; i = A[k] & B[k]
        js.append(len(i) / len(u) if u else 1.0)
    print(f"  mean Jaccard = {np.mean(js):.3f}  (expect ≈1.0 with cudnn.deterministic)")

# --------------------------------------------------------------------------------
# A.10 — Benchmark plot: fallback% (bar) + AUROC Δ (line), per config
# --------------------------------------------------------------------------------
labels = [str(r["label"]) for r in ROWS]
fbp    = [float(r["fallback_pct"]) for r in ROWS]
truep  = [float(r["true_causal_pct"]) for r in ROWS]
leakp  = [float(r["leak_pct"]) for r in ROWS]
dauc   = [float(r["macroAUROC"]) - FULL_AUROC for r in ROWS]

x = np.arange(len(labels)); w = 0.38
fig, ax1 = plt.subplots(figsize=(15, 6))
ax1.bar(x - w/2, truep, w, label="true causal %", color="#2ca02c")
ax1.bar(x + w/2, fbp,   w, label="fallback %",    color="#1f77b4")
ax1.plot(x, leakp, "o--", color="#d62728", lw=1.4, ms=4, label="leak %")
ax1.set_ylabel("percent"); ax1.set_ylim(0, 105)
ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=55, ha="right", fontsize=8)
ax1.axhline(30, color="#888", ls=":", lw=1)   # fallback health line
ax2 = ax1.twinx()
ax2.plot(x, dauc, "s-", color="#9467bd", lw=1.6, ms=5, label="Δ macroAUROC vs FULL")
ax2.set_ylabel("Δ macroAUROC", color="#9467bd"); ax2.axhline(0, color="#9467bd", ls=":", lw=0.8)
l1, la1 = ax1.get_legend_handles_labels(); l2, la2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, la1 + la2, loc="upper right", fontsize=9)
ax1.set_title(f"Patch-mining ablation · n={len(ABL_PLANS)} · seed={ABL_SEED} "
              f"(FULL AUROC={FULL_AUROC:.3f})")
plt.tight_layout(); plt.show()

_all_done = all(lab in done for lab in _ORDER)
print(f"\n{'✅ Ablations COMPLETE' if _all_done else '💾 PARTIAL — re-run to finish'} → {ABL_OUT_CSV}")
if not _all_done:
    print("   ↻ Re-run this cell: completed configs are skipped, the rest continue.")

# Visualization

In [ ]:
import pickle, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

with open("/kaggle/working/stage3_outputs.pkl", "rb") as f:
    rd = pickle.load(f)
results = rd["train"] + rd["val"]
TARGET_CLASSES = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]

def cstyle(p):   # causal color
    return ("#1e8cff", "--") if p.selection_source == "fallback" else ("#00dc3c", "-")

def pick_for(cls):
    true_c = [r for r in results if r.image_path and
              any(p.pathology == cls and p.selection_source != "fallback" for p in r.causal_patches)]
    any_c  = [r for r in results if r.image_path and
              any(p.pathology == cls for p in r.causal_patches)]
    return (true_c[0] if true_c else (any_c[0] if any_c else None)), bool(true_c)

fig, axes = plt.subplots(2, 3, figsize=(16, 11))
axes = axes.ravel()

for ax, cls in zip(axes, TARGET_CLASSES):
    r, has_true = pick_for(cls)
    if r is None:
        ax.set_title(f"{cls}\n(no image with this class)", fontsize=11)
        ax.axis("off"); continue

    img = Image.open(r.image_path).convert("RGB").resize((512, 512), Image.LANCZOS)
    ax.imshow(img); ax.axis("off")

    # causal + spur_in for THIS class; spur_out is class-agnostic background
    for p in r.causal_patches:
        if p.pathology != cls: continue
        col, ls = cstyle(p); x1, y1, x2, y2 = p.box
        ax.add_patch(mpatches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=col, lw=2.2, linestyle=ls))
        ax.text(x1+2, y1-4, f"{p.combined_score:.2f}", color=col, fontsize=8, weight="bold")
    for p in r.spurious_patches:
        if p.spurious_source == "in_anatomy" and p.pathology == cls:
            x1, y1, x2, y2 = p.box
            ax.add_patch(mpatches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="#dc1e1e", lw=1.8))
        elif p.spurious_source == "outside_anatomy":
            x1, y1, x2, y2 = p.box
            ax.add_patch(mpatches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="#ffa000", lw=1.6, linestyle=(0, (5, 4))))

    nc = sum(p.pathology == cls for p in r.causal_patches)
    tag = "true causal" if has_true else "FALLBACK only"
    ax.set_title(f"{cls}  ({nc} causal · {tag})\n{r.image_name}", fontsize=10)

# hide unused 6th panel, add one shared legend there
axes[5].axis("off")
axes[5].legend(handles=[
    mpatches.Patch(color="#00dc3c", label="causal (disease)"),
    mpatches.Patch(color="#1e8cff", label="causal fallback"),
    mpatches.Patch(color="#dc1e1e", label="spurious in-anatomy"),
    mpatches.Patch(color="#ffa000", label="spurious outside-anatomy"),
], loc="center", fontsize=13, title="patch type")

plt.tight_layout(); plt.show()

In [15]:
import pickle
from collections import defaultdict

rd  = pickle.load(open("/kaggle/working/stage3_outputs.pkl", "rb"))
res = rd["train"] + rd["val"]
CLASSES = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]

img_any  = defaultdict(set)   # images with >=1 causal patch of that class
img_true = defaultdict(set)   # images with >=1 TRUE (non-fallback) causal of that class
n_patch  = defaultdict(int)   # total causal patches of that class

for r in res:
    for p in r.causal_patches:
        if p.pathology in CLASSES:
            n_patch[p.pathology] += 1
            img_any[p.pathology].add(r.image_name)
            if p.selection_source != "fallback":
                img_true[p.pathology].add(r.image_name)

print(f"{'class':<18}{'imgs w/causal':>14}{'imgs w/TRUE':>13}{'causal patches':>16}")
print("-" * 61)
for c in CLASSES:
    print(f"{c:<18}{len(img_any[c]):>14}{len(img_true[c]):>13}{n_patch[c]:>16}")
print("-" * 61)
print(f"{'TOTAL (unique imgs)':<18}"
      f"{len(set().union(*img_any.values())) if img_any else 0:>14}"
      f"{len(set().union(*img_true.values())) if img_true else 0:>13}"
      f"{sum(n_patch.values()):>16}")

class              imgs w/causal  imgs w/TRUE  causal patches
-------------------------------------------------------------
Atelectasis                    5            5              22
Cardiomegaly                   5            5              25
Consolidation                  7            7              30
Edema                         11           11              55
Pleural Effusion              19           14              76
-------------------------------------------------------------
TOTAL (unique imgs)            29           25             208
